# Chapter 02 — Introduction to the Human Brain (with Python)

> **Repository:** [https://github.com/ArunimGuchait/neuroimaging-intro](https://github.com/ArunimGuchait/neuroimaging-intro)  
> **Open in Colab:** [Launch this notebook](https://colab.research.google.com/github/ArunimGuchait/neuroimaging-intro/blob/main/introduction_to_human_brain.ipynb)

This notebook introduces core neuroanatomy and imaging modalities, then demonstrates loading and visualizing a public anatomical MRI using Python (nilearn / nibabel).

**Prerequisite:** Chapter 01 (Python basics) is assumed; see [Chapter 01](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_python_for_neuroimaging.ipynb).

**Navigation:** [Chapter 01 — Python basics](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_python_for_neuroimaging.ipynb) · [Chapter 03 — Neuroimaging Analysis](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_neuroimaging_analysis.ipynb)

---

## How to run this notebook

- Run cells in order (top → bottom). This notebook assumes familiarity with basic Python covered in Chapter 01.
- To run locally: create and use a virtual environment named `neuro-env`, install the requirements, then open this notebook in Jupyter, Jupyter Lab, or VS Code.

**Local setup (recommended)**

Windows PowerShell:

```powershell
python -m venv neuro-env
.\neuro-env\Scripts\Activate.ps1
pip install -r requirements.txt
```

Windows (cmd):

```cmd
.\neuro-env\Scripts\activate
pip install -r requirements.txt
```

macOS / Linux:

```bash
python3 -m venv neuro-env
source neuro-env/bin/activate
pip install -r requirements.txt
```

### Google Colab

- Open in Colab with this direct link: https://colab.research.google.com/github/ArunimGuchait/neuroimaging-intro/blob/main/introduction_to_human_brain.ipynb
- This notebook includes a Colab-detection/setup cell (run it first) which will `pip install` required packages and optionally mount Google Drive and configure a persistent `nilearn` cache in Drive.
- Colab VM storage is ephemeral — mount Google Drive if you want to persist downloaded datasets across sessions.

In [ ]:
# Purpose: Colab-specific setup. Detects Colab, installs packages, mounts Drive,
# and sets a persistent nilearn cache directory inside Drive. Run this cell first on Colab.
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess
    print('Running on Google Colab — performing Colab-specific setup')
    # Install required packages (idempotent if already installed)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nilearn', 'nibabel', 'matplotlib', 'numpy'])
    # Mount Google Drive for persistent storage of nilearn cache
    from google.colab import drive
    drive.mount('/content/drive')
    drive_cache = '/content/drive/MyDrive/nilearn_cache'
    os.makedirs(drive_cache, exist_ok=True)
    # Configure nilearn to use the Drive-based cache directory
    os.environ['NILEARN_DATA'] = drive_cache
    print('Colab setup complete. Nilearn data directory:', drive_cache)
else:
    print('Not running in Colab; skip Colab-specific setup. Use the local `neuro-env` virtual environment.')

## Learning objectives
- Understand major brain divisions and common anatomical terms.
- Know principal neuroimaging modalities and what they measure.
- Load and visualize a standard T1-weighted anatomical MRI in Python.
- Inspect basic image metadata (shape, voxel size, orientation).

## Who is this for?
This chapter is written for readers new to neuroimaging. It focuses on intuition and practical Python examples you will use in later analysis chapters.

## Short anatomy primer — words you will see often

- Brain vs. skull vs. scalp: the brain is the soft tissue inside the skull. MRI visualizes brain tissue directly.
- Gray matter: neuronal cell bodies (cortex, subcortical nuclei).
- White matter: myelinated axon bundles connecting regions.

### Major structural divisions
- Cerebrum (cerebral hemispheres): the largest part—divided into lobes (frontal, parietal, temporal, occipital)—is responsible for high-level functions, including processing sensory information, initiating voluntary movement, and regulating complex cognitive processes like intelligence, memory, personality, and language. It interprets sensory input (sight, sound, touch) and controls conscious, purposeful behaviors.
- Cerebellum: at the back, involved in coordination and balance.
- Brainstem: connects brain to spinal cord; vital functions (breathing, heart rate).


![alt text](images/ch02/Gehirn,_lateral_-_Lobi_+_Stammhirn_+_Cerebellum_eng.svg.png)

- _Figure: Lateral view of a human brain, telencephalic lobes, cerebellum and brainstem colored._
- Source: https://en.wikipedia.org/wiki/Cerebellum#/media/File:Gehirn,_lateral_-_Lobi_+_Stammhirn_+_Cerebellum_eng.svg


### Common coordinate and orientation terms
- Axial (or transverse): slices from top to bottom.
- Sagittal: left-right slices (like seeing a midline cut).
- Coronal: front-back slices.
- Voxel: 3D pixel; images have `shape` (voxels) and `affine` mapping voxels to mm coordinates.

![alt text](images/ch02/brain-planes-sections.png)

- _Figure: Neuroanatomical Planes_
- Source: https://www.humanbiomedia.org/anatomical-planes-media/

![alt text](<images/ch02/Screenshot 2026-03-05 at 03-25-20 Instagram.png>)

- _Figure: Neuroanatomical Planes_
- Source: https://www.linkedin.com/posts/drrvs_magnetic-resonance-imaging-activity-7364899207696060416-Vfc8/

## Common neuroimaging modalities — a quick guide
- Structural MRI (T1-weighted): high-resolution anatomy — used for segmentation and normalization.
- Functional MRI (fMRI): measures BOLD signal changes over time — used to infer neural activity indirectly.
- Diffusion MRI (dMRI / DTI): measures water diffusion — used to reconstruct white-matter pathways.
- EEG / MEG: direct electrophysiological measures with high temporal but low spatial resolution.

![alt text](<images/ch02/ChatGPT Image Mar 5, 2026, 03_49_37 AM.png>)
- _Figure: Common neuroimaging techniques overview_
- Source: image created by ChatGPT

We'll demonstrate a structural (T1) MRI since it's the foundational image used in many workflows.

## Standard spaces and templates
Neuroimaging analyses usually transform individual brains into a 'standard space' so data from multiple subjects align. A common template is the MNI (Montreal Neurological Institute) space. We'll load an MNI-like T1 template for demonstration.

---
## Practical demo — load and visualize an anatomical MRI
The code below installs required packages (if missing), downloads a public anatomical template using `nilearn.datasets`, and visualizes slices. Run cells sequentially.


In [ ]:
# Purpose: Ensure required Python packages are available.
# This cell tries to import each package and installs it with pip if missing.
# Output: pip install logs if packages are installed, then a confirmation print.
# Run only once in a fresh environment.
import sys
import subprocess
def maybe_install(pkgs):
    for pkg in pkgs:
        try:
            __import__(pkg)
        except Exception:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

maybe_install(['nilearn','nibabel','matplotlib','numpy'])
print('Dependencies checked/installed')

In [ ]:
# Purpose: Import libraries used below (nilearn for datasets/plotting, nibabel for image I/O).
# Output: No printed output; errors indicate missing packages or import problems.
import os
from nilearn import datasets, plotting, image
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Purpose: Download a public T1-weighted anatomical template (ICBM152).
# Output: a local filesystem path to the downloaded template is printed.
# nilearn caches files under the user cache directory (usually ~/.cache/nilearn).
template = datasets.fetch_icbm152_2009()
anat_path = template['t1']  # local path to the template image
print('Template path:', anat_path)

In [ ]:
# Purpose: Load the anatomical image and inspect metadata (shape, affine, voxel sizes).
# Output: prints the image shape (in voxels), the affine matrix mapping voxels->mm,
# and approximate voxel sizes in millimeters.
img = nib.load(anat_path)
data = img.get_fdata()
print('Image shape (voxels):', data.shape)
print('Affine (voxel->mm:\n', img.affine)
voxel_sizes = np.sqrt((img.affine[:3,:3] ** 2).sum(axis=0))
print('Approx voxel sizes (mm):', voxel_sizes)

In [ ]:
# Purpose: Plot a histogram of voxel intensities to understand intensity distribution.
# Output: a matplotlib histogram; helps identify intensity ranges for tissue contrast.
flat = data.ravel()
flat = flat[np.isfinite(flat)]
plt.figure(figsize=(6,3))
plt.hist(flat, bins=100, color='gray')
plt.title('Intensity distribution (template)')
plt.xlabel('Intensity')
plt.ylabel('Voxel count')
plt.show()

In [ ]:
# Purpose: Show orthogonal anatomical slices (axial, sagittal, coronal).
# Output: an interactive figure in the notebook with the template overlaid;
# edges highlight boundaries and can help orient anatomy.
display = plotting.plot_anat(anat_path, title='ICBM152 T1 template (orthogonal views)')
display.add_edges(anat_path)
plotting.show()

## What we just did
- Installed necessary Python packages (if needed).
- Downloaded a public anatomical template via `nilearn.datasets.fetch_icbm152_2009()`.
- Inspected image `shape`, `affine`, and `voxel` sizes — fundamentals you will need when aligning or resampling images.
- Plotted slices and an intensity histogram to build intuition about the image contents.

## Next concepts (covered in later chapters)
- Registration and normalization (aligning subjects to a template).
- Brain extraction / skull-stripping and tissue segmentation.
- Functional preprocessing and statistical analysis — see [Chapter 03: introduction_neuroimaging_analysis.ipynb](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_neuroimaging_analysis.ipynb) for preprocessing and basic analyses, and [Chapter 04: task_based_fmri_analysis.ipynb](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/task_based_fmri_analysis.ipynb) for task-based GLM/statistics.

This notebook intentionally keeps the demo simple so you can run it quickly and focus on anatomical intuition.

## References and resources
- Nilearn documentation: https://nilearn.github.io/
- ICBM152 template reference: see `nilearn.datasets.fetch_icbm152_2009` documentation.
- Recommended reading: Beginner neuroanatomy texts or online visual atlases for more anatomy detail.
- Additional curated fMRI resources: see the companion repository 'fmri-analysis-resources' for tutorials, videos, and tool-specific guides (FSL, fMRIPrep, Nilearn, NiBabel): https://github.com/ArunimGuchait/fmri-analysis-resources

---
## Try it locally
1. Create and activate the recommended virtual environment named `neuro-env` (see the 'How to run' cell).
2. Open this notebook and run cells top-to-bottom. If downloads fail, ensure you have internet access and the environment can run `pip` installs.
3. When ready, proceed to `Chapter 03: introduction_neuroimaging_analysis.ipynb` for preprocessing and analysis workflows.